# Videovergleich für Ablauf2.1

Dieses Notebook verwendet den Trace `Ablauf2.1 08072026` aus `event_log.xes`. Die Liste der sichtbaren Events ist bewusst manuell gepflegt: Sie enthält nur Vorgänge, die in einer Fabrikaufnahme anhand von Bewegung, Stillstand, Ankunft oder sichtbarer Handhabung plausibel markiert werden können.

Die Zeitbasis des Logs bleibt UTC. Das erste `fts_driving_start`-Event dient als Baseline und wird auf Videozeit 0:39 gesetzt; die erwarteten Videozeiten der übrigen Events ergeben sich aus ihrer Zeitdifferenz zu diesem Ereignis.

In [1]:
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

def find_workspace_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "default_process").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root (expected a default_process folder) "
        f"above the current directory: {start}"
    )


WORKSPACE_ROOT = find_workspace_root()
VIDEO_DIR = WORKSPACE_ROOT / "video_time_delay_comparison"
XES_PATH = VIDEO_DIR / "event_log.xes"
TRACE_NAME = "Ablauf2.1 08072026"
XES_NAMESPACE = {"xes": "http://www.xes-standard.org/"}


def read_xes_trace(path: Path, trace_name: str) -> list[dict[str, str]]:
    root = ET.parse(path).getroot()
    for trace in root.findall("xes:trace", XES_NAMESPACE):
        trace_attributes = {
            child.attrib["key"]: child.attrib.get("value", "")
            for child in trace.findall("xes:string", XES_NAMESPACE)
        }
        if trace_attributes.get("concept:name") != trace_name:
            continue

        events = []
        for event in trace.findall("xes:event", XES_NAMESPACE):
            attributes = {}
            for child in event:
                key = child.attrib.get("key")
                if key:
                    attributes[key] = child.attrib.get("value", "")
            events.append(attributes)
        return events

    raise ValueError(f"Trace nicht gefunden: {trace_name}")


raw_events = read_xes_trace(XES_PATH, TRACE_NAME)
events = pd.DataFrame(raw_events)
events["timestamp"] = pd.to_datetime(events["time:timestamp"], utc=True)
events = events.sort_values("timestamp").reset_index(drop=True)

# Der XES-Export enthält je nach Quelle JSON- und JSONL-Wiederholungen.
events = events[events["source_file"].str.endswith(".jsonl", na=False)].copy()
events["details_dict"] = events["details"].apply(
    lambda value: json.loads(value) if isinstance(value, str) and value else {}
)
events["relative_s"] = (
    events["timestamp"] - events["timestamp"].min()
).dt.total_seconds()

event_counts = events["concept:name"].value_counts()
print(f"{len(events):,} JSONL-Events im Trace {TRACE_NAME!r}")
print(f"Zeitraum: {events['timestamp'].min()} bis {events['timestamp'].max()}")
print("\nHäufigste Eventtypen:")
print(event_counts.head(20).to_string())

2,542 JSONL-Events im Trace 'Ablauf2.1 08072026'
Zeitraum: 2026-07-08 08:02:36.500000+00:00 bis 2026-07-08 08:33:33.463000+00:00

Häufigste Eventtypen:
concept:name
ORDER_RECEIVED                             757
ORDER_STARTED                              757
ORDER_STOPPED                              171
FTS_clearLoadHandler                        64
MODULE_ORDER_DROP                           41
MODULE_DROP_FINISHED                        41
MODULE_ORDER_PICK                           34
MODULE_PICK_FINISHED                        34
MODULE_DROP_RUNNING                         32
MODULE_PICK_RUNNING                         32
MODULE_CHECK_QUALITY_RUNNING                27
ERROR_UNKNOWN                               24
MODULE_RGB_NFC_FINISHED                     18
ORDER_REQUEST                               18
ORDER_RESPONSE                              18
FTS_ORDER_SENT_TO_SVR4H73901                18
FTS_LOAD_HANDLING_STARTED_AT_SVR4H73901     18
FTS_LOAD_HANDLING_STOPPED_AT_SVR4H73

## Manuelle Auswahl visuell erkennbarer Events

Jedes ausgewählte Log-Event besitzt genau **einen Timestamp**. Eine Fahrt ist die einzige Ausnahme auf Prozessebene: Sie besteht aus zwei einzelnen Events, `FTS_DRIVING_STARTED_FROM_*` (Start) und `FTS_DRIVING_STOPPED_AT_*` (Ende). Die Fahrtdauer wird erst beim Vergleich aus diesen beiden Zeitpunkten berechnet.

| Eventfamilie | Log-Event bzw. Zeitpunkt | Im Video erkennbar? | Verwendung |
|---|---|---:|---|
| FTS fährt los | `FTS_DRIVING_STARTED_FROM_*` | Ja | Einzelner Startzeitpunkt einer Fahrt |
| FTS hält an | `FTS_DRIVING_STOPPED_AT_*` | Ja | Einzelner Endzeitpunkt derselben Fahrt |
| Einlagerung im HBW | Positiver `loadTimestamp` aus `HBW state.loads[]` | Ja | Einlagerungszeitpunkt am Lagerplatz; `0` bedeutet leer, wiederholte Zustände werden dedupliziert |
| PICK | `MODULE_PICK_FINISHED` | Ja | Einzelner Zeitpunkt des abgeschlossenen PICKs |
| DROP | `MODULE_DROP_FINISHED` | Ja | Einzelner Zeitpunkt des abgeschlossenen DROPs |
| Qualitätsprüfung | `MODULE_CHECK_QUALITY_*` | Eher nein | Interner Zustand, ohne sichtbares Ergebnis nicht sicher markierbar |
| Auftrag/MQTT | `ORDER_*`, `FTS_ORDER_SENT_*` | Nein | Interne Kommunikations- oder Zustandsereignisse |

Identische Export-Duplikate mit gleichem Eventnamen, Timestamp und Ziel werden vor der Auswertung entfernt.

In [2]:
VISIBLE_EVENT_DEFINITIONS = {
    "fts_driving_start": {
        "pattern": r"^FTS_DRIVING_STARTED_FROM_",
        "label": "FTS fährt los",
        "kind": "driving_start",
    },
    "fts_driving_end": {
        "pattern": r"^FTS_DRIVING_STOPPED_AT_",
        "label": "FTS hält an",
        "kind": "driving_end",
    },
    "pick": {
        "pattern": r"^MODULE_PICK_FINISHED$",
        "label": "PICK abgeschlossen",
        "kind": "point",
    },
    "drop": {
        "pattern": r"^MODULE_DROP_FINISHED$",
        "label": "DROP abgeschlossen",
        "kind": "point",
    },
}

HBW_JSONL_PATH = WORKSPACE_ROOT / "default_process" / "Ablauf2.1 08072026.jsonl"
if not HBW_JSONL_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {HBW_JSONL_PATH.name} under "
        f"{WORKSPACE_ROOT / 'default_process'}"
    )

HBW_SERIAL_NUMBER = "SVR4H73901"


def hbw_load_event_rows(event_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    with HBW_JSONL_PATH.open(encoding="utf-8") as source:
        for line in source:
            record = json.loads(line)
            if record.get("topic") != f"module/v1/ff/{HBW_SERIAL_NUMBER}/state":
                continue

            for load in record.get("payload", {}).get("loads", []):
                load_timestamp = load.get("loadTimestamp")
                if not load_timestamp:
                    continue

                timestamp = pd.to_datetime(load_timestamp, unit="ms", utc=True)
                rows.append(
                    {
                        "family": "hbw_load",
                        "label": "Einlagerung im HBW",
                        "kind": "point",
                        "event_name": "HBW_LOAD_TIMESTAMP",
                        "log_timestamp": timestamp,
                        "log_time_s": (
                            timestamp - event_frame["timestamp"].min()
                        ).total_seconds(),
                        "location": load.get("loadPosition", ""),
                        "destination": HBW_SERIAL_NUMBER,
                        "load_type": load.get("loadType", ""),
                        "load_id": load.get("loadId", ""),
                    }
                )

    return pd.DataFrame(rows).drop_duplicates(
        subset=["log_timestamp", "location", "load_id"]
    )


def visible_event_rows(event_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for family, definition in VISIBLE_EVENT_DEFINITIONS.items():
        matching_events = event_frame[
            event_frame["concept:name"].str.match(definition["pattern"], na=False)
        ].sort_values("timestamp")

        for _, event in matching_events.drop_duplicates(
            subset=["concept:name", "timestamp", "location", "destination_node"]
        ).iterrows():
            rows.append(
                {
                    "family": family,
                    "label": definition["label"],
                    "kind": definition["kind"],
                    "event_name": event["concept:name"],
                    "log_timestamp": event["timestamp"],
                    "log_time_s": event["relative_s"],
                    "location": event.get("location", ""),
                    "destination": event.get("destination_node", ""),
                    "load_type": "",
                    "load_id": "",
                }
            )

    visible = pd.DataFrame(rows)
    visible = pd.concat(
        [visible, hbw_load_event_rows(event_frame)], ignore_index=True
    ).sort_values("log_timestamp").reset_index(drop=True)
    visible["occurrence"] = visible.groupby("family").cumcount() + 1
    return visible


visible_events = visible_event_rows(events)

driving_start_events = visible_events.loc[
    visible_events["family"].eq("fts_driving_start"), "log_timestamp"
]
if driving_start_events.empty:
    raise ValueError("Kein fts_driving_start-Event als Videozeit-Baseline gefunden")

video_baseline_timestamp = driving_start_events.min()
visible_events["expected_video_time"] = (
    pd.to_timedelta(39, unit="s")
    + visible_events["log_timestamp"]
    - video_baseline_timestamp
)

output_path = VIDEO_DIR / "visible_events.csv"
csv_created = not output_path.exists()
if csv_created:
    with output_path.open("x", encoding="utf-8-sig", newline="") as output_file:
        visible_events.to_csv(output_file, index=False)

table_columns = [
    "family", "kind", "occurrence", "event_name", "log_timestamp", "log_time_s",
    "location", "destination", "load_type", "load_id", "expected_video_time",
]
excel_events = visible_events[table_columns].copy()
excel_events["log_timestamp"] = excel_events["log_timestamp"].dt.tz_convert(None)

excel_path = VIDEO_DIR / "visible_events.xlsx"
excel_created = not excel_path.exists()
if excel_created:
    with excel_path.open("xb") as output_file:
        excel_events.to_excel(
            output_file, index=False, sheet_name="Visible Events", engine="openpyxl"
        )

    from openpyxl import load_workbook
    from openpyxl.worksheet.table import Table, TableStyleInfo

    workbook = load_workbook(excel_path)
    worksheet = workbook["Visible Events"]
    table = Table(displayName="VisibleEvents", ref=worksheet.dimensions)
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )
    worksheet.add_table(table)
    worksheet.freeze_panes = "A2"
    for cells in worksheet.iter_cols():
        if cells[0].value == "log_timestamp":
            for cell in cells[1:]:
                cell.number_format = 'yyyy-mm-dd hh:mm:ss.000 "UTC"'
        elif cells[0].value == "expected_video_time":
            for cell in cells[1:]:
                cell.number_format = "[m]:ss.000"
    workbook.save(excel_path)

print(f"{len(visible_events):,} sichtbare Einzelereignisse gefunden")
csv_status = "erstellt" if csv_created else "bereits vorhanden, nicht überschrieben"
excel_status = "erstellt" if excel_created else "bereits vorhanden, nicht überschrieben"
print(f"CSV {csv_status}: {output_path.resolve()}")
print(f"Excel-Tabelle {excel_status}: {excel_path.resolve()}")

visible_events[table_columns]

197 sichtbare Einzelereignisse gefunden
CSV erstellt: C:\Users\olive\Documents\Uni\Forschungsprojekt\video_time_delay_comparison\visible_events.csv
Excel-Tabelle erstellt: C:\Users\olive\Documents\Uni\Forschungsprojekt\video_time_delay_comparison\visible_events.xlsx


,family,kind,occurrence,event_name,log_timestamp,log_time_s,location,destination,load_type,load_id,expected_video_time
0,fts_driving_start,driving_start,1,FTS_DRIVING_STARTED_FROM_SVR4H92774_TO_SVR4H73901,2026-07-08 08:03:11.096000+00:00,34.596,SVR4H92774,SVR4H73901,,,0 days 00:00:39
1,fts_driving_end,driving_end,1,FTS_DRIVING_STOPPED_AT_SVR4H92774_FROM_SVR4H73901,2026-07-08 08:03:12.508000+00:00,36.008,SVR4H92774,SVR4H73901,,,0 days 00:00:40.412000
2,drop,point,1,MODULE_DROP_FINISHED,2026-07-08 08:03:26.237000+00:00,49.737,NaN,NaN,,,0 days 00:00:54.141000
3,fts_driving_start,driving_start,2,FTS_DRIVING_STARTED_FROM_SVR4H92774_TO_SVR4H73901,2026-07-08 08:03:27.531000+00:00,51.031,SVR4H92774,SVR4H73901,,,0 days 00:00:55.435000
4,fts_driving_end,driving_end,2,FTS_DRIVING_STOPPED_AT_SVR4H73901_FROM_SVR4H73901,2026-07-08 08:03:38.391000+00:00,61.891,SVR4H73901,SVR4H73901,,,0 days 00:01:06.295000
...,...,...,...,...,...,...,...,...,...,...,...
192,fts_driving_end,driving_end,61,FTS_DRIVING_STOPPED_AT_SVR4H77011_FROM_SVR4H92774,2026-07-08 08:32:46.398000+00:00,1809.898,SVR4H77011,SVR4H92774,,,0 days 00:30:14.302000
193,drop,point,32,MODULE_DROP_FINISHED,2026-07-08 08:32:54.425000+00:00,1817.925,NaN,NaN,,,0 days 00:30:22.329000
194,fts_driving_start,driving_start,62,FTS_DRIVING_STARTED_FROM_SVR4H77011_TO_SVR4H92774,2026-07-08 08:32:54.690000+00:00,1818.190,SVR4H77011,SVR4H92774,,,0 days 00:30:22.594000
195,fts_driving_end,driving_end,62,FTS_DRIVING_STOPPED_AT_SVR4H92774_FROM_SVR4H92774,2026-07-08 08:33:04.748000+00:00,1828.248,SVR4H92774,SVR4H92774,,,0 days 00:30:32.652000
